# 02 — Treino BERTimbau + K-Fold Estratificado
**NurseXAI** | Fine-tuning e calibração de thresholds

---
**Requer:** `01_data_analysis.ipynb` executado  
**Executar antes de:** `03_evaluation.ipynb`

Este notebook realiza:
1. Configuração de hiperparâmetros
2. Fine-tuning BERTimbau (holdout)
3. K-Fold estratificado + calibração de thresholds
4. Guardar modelo e thresholds calibrados

## 1. Setup

In [ ]:
import os, json, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizerFast, BertModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

## 2. Configuração Central

In [ ]:
CONFIG = {
    'model_name'          : 'neuralmind/bert-base-portuguese-cased',
    'max_length'          : 256,
    'batch_size'          : 8,
    'epochs'              : 20,
    'learning_rate'       : 2e-5,
    'dropout'             : 0.3,
    'test_size'           : 0.20,
    'val_size'            : 0.10,
    'warmup_ratio'        : 0.1,
    'weight_decay'        : 0.01,
    'early_stop_patience' : 5,
}

SAVE_PATH = '../outputs/nursexai_bert_best.pt'
os.makedirs('../outputs', exist_ok=True)

print("✅ CONFIG carregado")
for k, v in CONFIG.items():
    print(f"   {k:<25}: {v}")

## 3. Carregar Dados

In [ ]:
df         = pd.read_csv('../data/notas_processadas.csv', encoding='utf-8-sig')
labels_bin = np.load('../data/labels_bin.npy')

with open('../data/label_cols.json', encoding='utf-8') as f:
    label_cols = json.load(f)

textos = df['nota_processada'].values
print(f"✅ {len(textos)} notas | {len(label_cols)} labels")

## 4. Split Treino / Validação / Teste

In [ ]:
# IMPORTANTE: não alterar random_state — garante reprodutibilidade
X_tv, X_test, Y_tv, Y_test = train_test_split(
    textos, labels_bin, test_size=CONFIG['test_size'], random_state=SEED
)
val_ratio = CONFIG['val_size'] / (1 - CONFIG['test_size'])
X_train, X_val, Y_train, Y_val = train_test_split(
    X_tv, Y_tv, test_size=val_ratio, random_state=SEED
)
Y_true_test = Y_test.copy()

print(f"✅ Splits (random_state={SEED})")
print(f"   Treino    : {len(X_train)}")
print(f"   Validação : {len(X_val)}")
print(f"   Teste     : {len(X_test)}")

## 5. Dataset e DataLoader

In [ ]:
class NotasEnfermagemDataset(Dataset):
    """Dataset PyTorch — tokeniza internamente no __getitem__."""
    def __init__(self, textos, labels, tokenizer, max_length=256):
        self.textos     = list(textos)
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.textos[idx],
            max_length    = self.max_length,
            padding       = 'max_length',
            truncation    = True,
            return_tensors= 'pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels'        : torch.FloatTensor(self.labels[idx])
        }

tokenizer     = BertTokenizerFast.from_pretrained(CONFIG['model_name'])
train_dataset = NotasEnfermagemDataset(X_train, Y_train, tokenizer, CONFIG['max_length'])
val_dataset   = NotasEnfermagemDataset(X_val,   Y_val,   tokenizer, CONFIG['max_length'])
test_dataset  = NotasEnfermagemDataset(X_test,  Y_true_test, tokenizer, CONFIG['max_length'])
train_loader  = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False)
test_loader   = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'], shuffle=False)

print(f"✅ DataLoaders prontos")
print(f"   Treino    : {len(train_loader)} batches")
print(f"   Validação : {len(val_loader)} batches")

## 6. Arquitectura NurseXAIBERT

In [ ]:
class NurseXAIBERT(nn.Module):
    """
    BERTimbau fine-tuned para classificação multi-label NANDA-I.
    Arquitectura: BertModel → CLS token → Dropout → Linear(768, 15)
    """
    def __init__(self, model_name, num_labels, dropout=0.3):
        super().__init__()
        self.bert       = BertModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls     = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

model = NurseXAIBERT(CONFIG['model_name'], len(label_cols), CONFIG['dropout']).to(device)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Modelo: {total:,} parâmetros ({trainable:,} treináveis)")

## 7. Treino Holdout

In [ ]:
# Loss com pos_weight para compensar desequilíbrio de labels
freq      = Y_train.sum(axis=0)
pos_w     = torch.tensor((len(Y_train) - freq) / (freq + 1e-6), dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

total_steps = len(train_loader) * CONFIG['epochs']
optimizer   = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = int(total_steps * CONFIG['warmup_ratio']),
    num_training_steps = total_steps
)

historico = {'train_loss': [], 'val_loss': [], 'val_f1': []}
melhor_f1, melhor_epoch, sem_melhoria = 0.0, 0, 0

print(f"{'Época':>6} | {'Loss Treino':>11} | {'Loss Val':>9} | {'F1 Val':>7}")
print("─" * 50)

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    loss_tr = 0.0
    for batch in train_loader:
        lbs = batch['labels'].to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch['input_ids'].to(device), batch['attention_mask'].to(device)), lbs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        loss_tr += loss.item()
    loss_tr /= len(train_loader)

    model.eval()
    loss_val, preds_v, labs_v = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            lbs    = batch['labels'].to(device)
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss_val += criterion(logits, lbs).item()
            preds_v.append((torch.sigmoid(logits) >= 0.5).int().cpu().numpy())
            labs_v.append(lbs.cpu().numpy())
    loss_val /= len(val_loader)
    val_f1    = f1_score(np.vstack(labs_v), np.vstack(preds_v), average='macro', zero_division=0)

    historico['train_loss'].append(loss_tr)
    historico['val_loss'].append(loss_val)
    historico['val_f1'].append(val_f1)

    melhor = val_f1 > melhor_f1
    if melhor:
        melhor_f1, melhor_epoch, sem_melhoria = val_f1, epoch, 0
        torch.save(model.state_dict(), SAVE_PATH)
    else:
        sem_melhoria += 1

    print(f"  {epoch:4d} {'★' if melhor else ' '} | {loss_tr:.4f} | {loss_val:.4f} | {val_f1:.4f}")

    if sem_melhoria >= CONFIG['early_stop_patience']:
        print(f"\n⏹  Early stopping na época {epoch}")
        break

print(f"\n✅ Melhor: época {melhor_epoch} | F1-val: {melhor_f1:.4f}")
model.load_state_dict(torch.load(SAVE_PATH))

## 8. K-Fold Estratificado + Calibração de Thresholds

In [ ]:
def calibrar_thresholds(probs_val, labels_val, label_names):
    """Grid search de threshold óptimo por label (F1-maximização)."""
    thresholds = {}
    for idx, label in enumerate(label_names):
        melhor_t, melhor_f1 = 0.5, 0.0
        for t in np.arange(0.10, 0.91, 0.05):
            preds = (probs_val[:, idx] >= t).astype(int)
            f1    = f1_score(labels_val[:, idx], preds, zero_division=0)
            if f1 > melhor_f1:
                melhor_f1, melhor_t = f1, t
        thresholds[label] = round(float(melhor_t), 2)
    return thresholds

N_SPLITS, EPOCHS_FOLD, PACIENCIA = 5, 15, 3
mskf = MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

ds_test_kf     = NotasEnfermagemDataset(list(X_test), Y_true_test, tokenizer, CONFIG['max_length'])
loader_test_kf = DataLoader(ds_test_kf, batch_size=CONFIG['batch_size'], shuffle=False)

resultados_folds, thresholds_por_fold = [], []

for fold_idx, (idx_tr, idx_val) in enumerate(mskf.split(X_tv, Y_tv)):
    print(f"\n{'='*50}\n  FOLD {fold_idx+1}/{N_SPLITS} | treino: {len(idx_tr)} | val: {len(idx_val)}\n{'='*50}")

    X_f_tr,  Y_f_tr  = [X_tv[i] for i in idx_tr],  Y_tv[idx_tr]
    X_f_val, Y_f_val = [X_tv[i] for i in idx_val], Y_tv[idx_val]

    ld_tr  = DataLoader(NotasEnfermagemDataset(X_f_tr,  Y_f_tr,  tokenizer, CONFIG['max_length']), batch_size=CONFIG['batch_size'], shuffle=True)
    ld_val = DataLoader(NotasEnfermagemDataset(X_f_val, Y_f_val, tokenizer, CONFIG['max_length']), batch_size=CONFIG['batch_size'], shuffle=False)

    model_f   = NurseXAIBERT(CONFIG['model_name'], len(label_cols), CONFIG['dropout']).to(device)
    freq_f    = Y_f_tr.sum(axis=0)
    pos_w_f   = torch.tensor((len(Y_f_tr) - freq_f) / (freq_f + 1e-6), dtype=torch.float32).to(device)
    crit_f    = nn.BCEWithLogitsLoss(pos_weight=pos_w_f)
    steps_f   = len(ld_tr) * EPOCHS_FOLD
    opt_f     = AdamW(model_f.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    sched_f   = get_linear_schedule_with_warmup(opt_f, int(steps_f * CONFIG['warmup_ratio']), steps_f)

    melhor_hl, melhor_state, sem_imp = float('inf'), None, 0
    for epoch in range(1, EPOCHS_FOLD + 1):
        model_f.train()
        for batch in ld_tr:
            lbs = batch['labels'].to(device)
            opt_f.zero_grad()
            loss = crit_f(model_f(batch['input_ids'].to(device), batch['attention_mask'].to(device)), lbs)
            loss.backward()
            nn.utils.clip_grad_norm_(model_f.parameters(), 1.0)
            opt_f.step(); sched_f.step()

        model_f.eval()
        pv, lv = [], []
        with torch.no_grad():
            for batch in ld_val:
                logits = model_f(batch['input_ids'].to(device), batch['attention_mask'].to(device))
                pv.append(torch.sigmoid(logits).cpu().numpy())
                lv.append(batch['labels'].numpy())
        pv, lv = np.vstack(pv), np.vstack(lv)
        hl_v   = hamming_loss(lv, (pv >= 0.5).astype(int))
        f1_v   = f1_score(lv, (pv >= 0.5).astype(int), average='macro', zero_division=0)
        print(f"  Ep {epoch:2d}/{EPOCHS_FOLD}  HL: {hl_v:.4f}  F1: {f1_v:.4f}")

        if hl_v < melhor_hl:
            melhor_hl, melhor_state, sem_imp = hl_v, copy.deepcopy(model_f.state_dict()), 0
        else:
            sem_imp += 1
            if sem_imp >= PACIENCIA:
                print(f"  ⏹  Early stopping época {epoch}")
                break

    model_f.load_state_dict(melhor_state)
    model_f.eval()

    pvc, lvc = [], []
    with torch.no_grad():
        for batch in ld_val:
            logits = model_f(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            pvc.append(torch.sigmoid(logits).cpu().numpy())
            lvc.append(batch['labels'].numpy())
    thresh_fold = calibrar_thresholds(np.vstack(pvc), np.vstack(lvc), label_cols)
    thresholds_por_fold.append(thresh_fold)

    ptc, ltc = [], []
    with torch.no_grad():
        for batch in loader_test_kf:
            logits = model_f(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            ptc.append(torch.sigmoid(logits).cpu().numpy())
            ltc.append(batch['labels'].numpy())
    ptc, ltc = np.vstack(ptc), np.vstack(ltc)
    preds_tc  = np.zeros_like(ptc, dtype=int)
    for idx, lbl in enumerate(label_cols):
        preds_tc[:, idx] = (ptc[:, idx] >= thresh_fold.get(lbl, 0.5)).astype(int)

    hl_f  = hamming_loss(ltc, preds_tc)
    f1m_f = f1_score(ltc, preds_tc, average='macro', zero_division=0)
    f1i_f = f1_score(ltc, preds_tc, average='micro', zero_division=0)
    resultados_folds.append({'fold': fold_idx+1, 'hl': hl_f, 'f1_macro': f1m_f, 'f1_micro': f1i_f})
    print(f"\n  ✅ Fold {fold_idx+1} | HL: {hl_f:.4f}  F1-macro: {f1m_f:.4f}")

f1s = [r['f1_macro'] for r in resultados_folds]
hls = [r['hl'] for r in resultados_folds]
print(f"\n{'='*50}")
print(f"Média F1-macro : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"Média HL       : {np.mean(hls):.4f} ± {np.std(hls):.4f}")

THRESHOLDS_FINAIS = {
    lbl: round(float(np.mean([t[lbl] for t in thresholds_por_fold])), 2)
    for lbl in label_cols
}
print("\n── Thresholds K-Fold médios finais ──")
for lbl, t in THRESHOLDS_FINAIS.items():
    print(f"  {lbl[:50]:<50}  {t:.2f}")

## 9. Guardar Artefactos

In [ ]:
with open('../data/thresholds_kfold_finais.json', 'w', encoding='utf-8') as f:
    json.dump(THRESHOLDS_FINAIS, f, ensure_ascii=False, indent=2)

pd.DataFrame(resultados_folds).to_csv('../outputs/kfold_results.csv', index=False)

torch.save({
    'model_state_dict': model.state_dict(),
    'config'          : CONFIG,
    'label_cols'      : label_cols,
    'thresholds'      : THRESHOLDS_FINAIS,
    'melhor_epoch'    : melhor_epoch,
    'melhor_f1_val'   : melhor_f1,
}, '../outputs/nursexai_checkpoint.pt')

print("✅ Guardado:")
print("   data/thresholds_kfold_finais.json")
print("   outputs/kfold_results.csv")
print("   outputs/nursexai_checkpoint.pt")
print("\n→ Avançar para: 03_evaluation.ipynb")